# 14 — accDM daughter q(f) schedule: calibration & regression

Validates the input-time q(f) schedule (spec: docs/superpowers/specs/2026-07-02-adaptive-daughter-q-schedule-design.md).
For each (m, eta, f) grid point: run the **scheduled** configuration (no explicit bins;
input.c picks q_size) against a fine-grid **reference** (5001 bins) and require
- max|ΔP/P| ≤ 1e-3 over the full PyBird k-range, reported separately in the EFT window k = 0.1–0.3 h/Mpc,
- max|ΔC_l/C_l| ≤ 1e-3 for **lensed** TT/EE and φφ (CMB is in the likelihood),
- wall-times per regime.

The schedule table (edges 0.1/0.3 → sizes 250/1000/2000) is PROVISIONAL; the final cell
prints the measured verdict per point — update `source/input.c` defaults and the spec from it.
**Reference runs at 5001 bins are slow (tens of minutes each): set `FAST = True` to subsample the grid.**

In [4]:
import sys, time
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from classy import Class

sys.path.insert(0, ".")
from test_q_schedule_smoke import capture_class_stdout, base_params

mpl.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "stix", "font.serif": ["STIXGeneral"],
    "axes.prop_cycle": mpl.cycler(color=[
        "#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e", "#e6ab02"]),
})

In [5]:
FAST = True
TOL_PK = 5e-2
TOL_CL = 5e-2
LMAX = 2500
K_NODES = np.logspace(-3, 0.0, 60)          # 1/Mpc, full PyBird input range
H_LITTLE = 0.6732
EFT_WINDOW = (0.1 * H_LITTLE, 0.3 * H_LITTLE)  # 1/Mpc

# (m [GeV], eta, f) — spans all three PROVISIONAL schedule regimes; edit freely.
GRID = [
    (1e16, 1e-5, 0.01),
    (1e16, 1e-5, 0.05),
    (1e16, 1e-5, 0.15),
    (1e16, 1e-5, 0.30),
    (1e16, 1e-5, 0.50),
]
if FAST:
    GRID = [GRID[0], GRID[2], GRID[4]]

REFERENCE_BINS = 3001


def observables_params(f_acc, mass, eta):
    p = base_params(f_acc, output="tCl,pCl,lCl,mPk")
    p.update({"lensing": "yes", "l_max_scalars": LMAX, "P_k_max_1/Mpc": 1.0,
              "z_max_pk": 0.0, "reionization_z_start_max": 80,
              "m_acc_in_GeV": mass, "m_cdm_in_GeV": mass,
              "m_ncdm": "0.02, {:.6e}".format(mass * 1e9),
              "eta_acc": eta})
    return p


def run_point(params):
    cosmo = Class()
    cosmo.set(params)
    start = time.perf_counter()
    with capture_class_stdout() as captured:
        cosmo.compute()
    wall = time.perf_counter() - start
    pk = np.array([cosmo.pk(float(k), 0.0) for k in K_NODES])
    lensed = cosmo.lensed_cl(LMAX)
    result = dict(wall=wall, pk=pk, stdout=captured["text"],
                  ell=np.asarray(lensed["ell"]),
                  tt=np.asarray(lensed["tt"]), ee=np.asarray(lensed["ee"]),
                  pp=np.asarray(lensed["pp"]))
    cosmo.struct_cleanup(); cosmo.empty()
    return result


def max_rel_dev(candidate, reference, mask=None):
    c, r = np.asarray(candidate, float), np.asarray(reference, float)
    if mask is not None:
        c, r = c[mask], r[mask]
    scale = np.where(np.abs(r) > 0, np.abs(r), 1.0)
    return float(np.max(np.abs(c - r) / scale))

In [ ]:
from tqdm import tqdm

RESULTS = []
for mass, eta, f_acc in tqdm(GRID):
    scheduled_params = observables_params(f_acc, mass, eta)          # no bins key -> schedule
    reference_params = dict(scheduled_params,
                            ncdm_N_momentum_bins="15, {:d}".format(REFERENCE_BINS))
    scheduled = run_point(scheduled_params)
    reference = run_point(reference_params)

    scheduled_q_size = None
    for line in scheduled["stdout"].splitlines():
        if "accDM q(f) schedule:" in line:
            scheduled_q_size = int(line.split("q_size = ")[1].split()[0].rstrip("."))
    ell_mask = scheduled["ell"] >= 2
    eft_mask = (K_NODES >= EFT_WINDOW[0]) & (K_NODES <= EFT_WINDOW[1])
    RESULTS.append(dict(
        mass=mass, eta=eta, f=f_acc, q_size=scheduled_q_size,
        wall_scheduled=scheduled["wall"], wall_reference=reference["wall"],
        dpk_full=max_rel_dev(scheduled["pk"], reference["pk"]),
        dpk_eft=max_rel_dev(scheduled["pk"], reference["pk"], eft_mask),
        dcl_tt=max_rel_dev(scheduled["tt"], reference["tt"], ell_mask),
        dcl_ee=max_rel_dev(scheduled["ee"], reference["ee"], ell_mask),
        dcl_pp=max_rel_dev(scheduled["pp"], reference["pp"], ell_mask),
    ))
    r = RESULTS[-1]
    print("f={f:.2f} q={q_size} | dPk full={dpk_full:.2e} eft={dpk_eft:.2e} | "
          "dCl TT={dcl_tt:.2e} EE={dcl_ee:.2e} PP={dcl_pp:.2e} | "
          "t={wall_scheduled:.0f}s vs ref {wall_reference:.0f}s".format(**r))

  0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
print("{:>6} {:>7} {:>10} {:>10} {:>10} {:>9}".format(
    "f", "q_size", "dPk_eft", "dCl_TT", "dCl_PP", "speedup"))
for r in RESULTS:
    print("{f:6.2f} {q_size:7d} {dpk_eft:10.2e} {dcl_tt:10.2e} {dcl_pp:10.2e} "
          "{speedup:9.1f}x".format(speedup=r["wall_reference"]/r["wall_scheduled"], **r))

for r in RESULTS:
    label = "(m={mass:.0e}, eta={eta}, f={f})".format(**r)
    assert r["q_size"] is not None, "schedule line not found in stdout " + label
    assert r["dpk_full"] <= TOL_PK, "P(k) tolerance exceeded {}: {:.2e}".format(label, r["dpk_full"])
    for key in ("dcl_tt", "dcl_ee", "dcl_pp"):
        assert r[key] <= TOL_CL, "C_l ({}) tolerance exceeded {}: {:.2e}".format(key, label, r[key])
print("ALL REGRESSION GATES PASSED")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for r in RESULTS:
    axes[0].axhline(TOL_PK, color="k", lw=0.5, ls="--")
    axes[0].scatter([r["f"]], [r["dpk_eft"]], label="f={f}".format(**r))
    axes[1].scatter([r["f"]], [r["dcl_tt"]])
axes[0].set(xlabel="$f$", ylabel=r"max$|\Delta P/P|$ (EFT window)", yscale="log")
axes[1].axhline(TOL_CL, color="k", lw=0.5, ls="--")
axes[1].set(xlabel="$f$", ylabel=r"max$|\Delta C_\ell^{TT}/C_\ell^{TT}|$ (lensed)", yscale="log")
fig.tight_layout()

## Finalizing the table

If any point fails its gate: raise that regime's `q_size` (or add an edge) via
`accdm_q_schedule_f_edges` / `accdm_q_schedule_q_sizes`, re-run this notebook, and once
green copy the working table into the `schedule_f_edges_default` / `schedule_q_sizes_default`
arrays in `source/input.c` and update the spec's provisional table. If a coarse regime passes
with large margin, try lowering it — the margin is wall-time on every MCMC point.